# Day-first retrieval prototype

Цель эксперимента: проверить, можно ли искать по дневнику через day-first retrieval.

## 1. Imports and paths

Входы:

- `data/processed/days_md/*.md` — исходные дневные markdown-файлы;
- `data/processed/days_json/*.json` — structured daily JSON как day index.

Выходы эксперимента:

- пока ничего не сохраняем обязательно;
- сначала собираем `pandas.DataFrame` в памяти;
- позже можно сохранить `data/memory/day_index.jsonl`, `chunks.jsonl`, `embeddings.npy`.

In [1]:
from pathlib import Path 
import json
import re
from dotenv import load_dotenv
import os

import numpy as np
import pandas as pd

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [3]:
from gigachat import GigaChat

In [4]:
PROJECT_ROOT = Path("..").resolve()

DAYS_MD_DIR = PROJECT_ROOT / "data" / "processed" / "days_md"
DAYS_JSON_DIR = PROJECT_ROOT / "data" / "processed" / "days_json"

DAYS_MD_DIR, DAYS_JSON_DIR

(WindowsPath('C:/Users/79104/Data Science/Data-Science-/SoloProjects/DiaryLens/data/processed/days_md'),
 WindowsPath('C:/Users/79104/Data Science/Data-Science-/SoloProjects/DiaryLens/data/processed/days_json'))

In [5]:
md_files = sorted(DAYS_MD_DIR.glob("*.md"))
json_files = sorted(DAYS_JSON_DIR.glob("*.json"))

len(md_files), len(json_files)

(75, 75)

## 2. Load daily JSONs

Daily JSON здесь не считается полным пересказом дня.

Его роль — навигационный индекс:

- дата;
- week_id;
- важные моменты;
- tensions / wins;
- open questions;
- key quotes;
- short_summary.

In [6]:
def resolve_source_day_md(source_path: str) -> Path:
    path = Path(source_path)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path

def load_json(path:Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

daily_records = []

for path in json_files:
    try:
        data = load_json(path)
        data['_json_path'] = str(path)
        daily_records.append(data)
    except Exception as e:
        print(f"Error loading {path}: {e}")

len(daily_records)

75

In [7]:
daily_df = pd.DataFrame(daily_records)
daily_df.head()

,date,week_id,source_day_md,important_moments,wins,tensions,emotions,body_energy_signals,study_signals,ml_ds_signals,social_signals,decisions,open_questions,key_quotes,short_summary,_json_path
0,2026-03-29,2026-W14,data/processed/days_md/2026-03-29.md,[{'quote': 'начал проходить рут Акихи. Пока мн...,[],"[{'quote': 'Столько дел у меня накопилось x, ч...","[{'quote': 'особенно, когда мы начали играть в...",[],[],[],"[{'quote': 'Встреча с друзьями в рестике.', 'n...",[],[],[{'quote': 'Надо как-то выпутываться и постара...,"Автор провел день, занимаясь чтением манги, об...",C:\Users\79104\Data Science\Data-Science-\Solo...
1,2026-03-30,2026-W14,data/processed/days_md/2026-03-30.md,"[{'quote': 'Сегодня снова тсукихиме.', 'note':...",[],"[{'quote': 'не особо нравится, что главного ге...","[{'quote': 'очень приятное.', 'note': 'Оценка ...","[{'quote': 'упражнялся на турниках.', 'note': ...","[{'quote': 'Английский и ИБ готовил.', 'note':...",[],"[{'quote': 'У всех свои переживания.', 'note':...",[],[],"[{'quote': 'Здоровья, здоровья, здоровья и сил...","Автор активно провёл день, участвовал в хакато...",C:\Users\79104\Data Science\Data-Science-\Solo...
2,2026-03-31,2026-W14,data/processed/days_md/2026-03-31.md,"[{'quote': 'По унику закрыл срочные таски', 'n...",[{'quote': 'На второй паре уже сам хорошо напи...,[{'quote': 'На первой паре по англу я много ош...,"[{'quote': 'Рад, что в четвертой главе он снов...",[{'quote': 'Очень устал после физического труд...,[],[],"[{'quote': 'Пошли с Андреем на скалодром.', 'n...",[],[],"[{'quote': 'Рад, что в четвертой главе он снов...","Автор провёл насыщенный день, включая учёбу, ф...",C:\Users\79104\Data Science\Data-Science-\Solo...
3,2026-04-01,2026-W14,data/processed/days_md/2026-04-01.md,"[{'quote': 'Скоро будет много дел', 'note': 'П...",[{'quote': 'Тут я даже смог насладиться самим ...,"[{'quote': 'я боюсь его, ведь понимаю, что эти...","[{'quote': 'хорошо расстроился', 'note': 'Эмоц...","[{'quote': 'чувствовал себя немного вяло', 'no...","[{'quote': 'писать лекции по Царьковой', 'note...",[{'quote': 'даже смог насладиться самим процес...,[{'quote': 'Андрей выслал мне большое голосово...,"[{'quote': 'решил, что пока что переписывать л...",[{'quote': 'нужна ли сейчас полная запись лекц...,[{'quote': 'нужно как-то научиться наслаждатьс...,"Автор провёл активный учебный день, начав с по...",C:\Users\79104\Data Science\Data-Science-\Solo...
4,2026-04-02,2026-W14,data/processed/days_md/2026-04-02.md,"[{'quote': 'Встал сначала в 5, потом в 7 утра....","[{'quote': 'Сел за первую лабу Царьковой.', 'n...","[{'quote': 'Из-за боли вставал сначала в 5, по...","[{'quote': 'Хорошее состояние дня.', 'note': '...","[{'quote': 'боль в животе утром', 'note': 'Физ...","[{'quote': 'Сел за алгосы, потом посидел на хх...",[],[{'quote': 'Сказал маме о своих планах по учеб...,[{'quote': 'Решил продолжить читать Тсукихиме ...,[],"[{'quote': 'Здоровья, здоровья, здоровья и сил...",День начался с дискомфорта из-за боли в животе...,C:\Users\79104\Data Science\Data-Science-\Solo...


## 3. Daily JSON coverage check

Теперь проверяем, насколько daily JSON полезны как day index.

Daily JSON не обязан быть полным пересказом дня.  
Но он должен помогать retrieval находить нужный день.

Проверяем:

- есть ли `short_summary`;
- сколько важных моментов;
- сколько tensions / wins;
- сколько open questions;
- сколько key quotes;
- какие поля чаще всего пустые.

Если многие поля пустые, retrieval по day index будет слабее, и больше веса придётся давать raw chunks.

In [8]:
INDEX_FIELDS = [
    "important_moments",
    "wins",
    "tensions",
    "emotions",
    "body_energy_signals",
    "study_signals",
    "ml_ds_signals",
    "social_signals",
    "decisions",
    "open_questions",
    "key_quotes",
]


def count_items(value):
    if isinstance(value, list):
        return len(value)
    if value is None:
        return 0
    return 1


coverage_rows = []

for day in daily_records:
    row = {
        "date": day.get("date"),
        "week_id": day.get("week_id"),
        "has_short_summary": bool(str(day.get("short_summary", "")).strip()),
        "short_summary_len": len(str(day.get("short_summary", "")).strip()),
    }
    
    for field in INDEX_FIELDS:
        row[f"{field}_count"] = count_items(day.get(field))
    
    coverage_rows.append(row)

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df.head())

,date,week_id,has_short_summary,short_summary_len,important_moments_count,wins_count,tensions_count,emotions_count,body_energy_signals_count,study_signals_count,ml_ds_signals_count,social_signals_count,decisions_count,open_questions_count,key_quotes_count
0,2026-03-29,2026-W14,True,185,5,0,2,2,0,0,0,1,0,0,1
1,2026-03-30,2026-W14,True,135,10,0,3,2,2,2,0,1,0,0,1
2,2026-03-31,2026-W14,True,202,8,1,3,2,1,0,0,2,0,0,2
3,2026-04-01,2026-W14,True,259,11,1,3,2,2,3,1,1,1,1,1
4,2026-04-02,2026-W14,True,210,6,1,3,1,1,1,0,1,1,0,1


In [9]:
field_count_cols = [f"{field}_count" for field in INDEX_FIELDS]

field_stats = pd.DataFrame({
    "field": INDEX_FIELDS,
    "non_empty_days": [(coverage_df[f"{field}_count"] > 0).sum() for field in INDEX_FIELDS],
    "empty_days": [(coverage_df[f"{field}_count"] == 0).sum() for field in INDEX_FIELDS],
    "avg_items_per_day": [coverage_df[f"{field}_count"].mean() for field in INDEX_FIELDS],
})

field_stats.sort_values("non_empty_days", ascending=False)

,field,non_empty_days,empty_days,avg_items_per_day
0,important_moments,61,14,6.280000
10,key_quotes,60,15,1.160000
2,tensions,59,16,2.266667
3,emotions,59,16,1.760000
4,body_energy_signals,58,17,1.253333
7,social_signals,42,33,0.693333
1,wins,40,35,0.600000
5,study_signals,38,37,0.746667
8,decisions,36,39,0.560000
9,open_questions,19,56,0.280000


Daily JSON уже годится для первого retrieval-прототипа. А ml_ds_signals может быть редким, потому что я писал не столько про конкретные DS-действия, сколько про состояние вокруг учёбы/карьеры. Однако в будущем стоит так же улучшить данный extraction

## 4. Build day-level embedding_text

Теперь собираем `embedding_text` для каждого дня.

Это временный текст для embeddings, который помогает найти релевантный день.



In [10]:
TEXT_FIELDS_FOR_DAY_INDEX = [
    "important_moments",
    "wins",
    "tensions",
    "emotions",
    "body_energy_signals",
    "study_signals",
    "ml_ds_signals",
    "social_signals",
    "decisions",
    "open_questions",
    "key_quotes",
]


def item_to_text(item):
    if isinstance(item, str):
        return item.strip()
    
    if isinstance(item, dict):
        parts = []
        
        quote = str(item.get("quote", "")).strip()
        note = str(item.get("note", "")).strip()
        
        if quote:
            parts.append(f"quote: {quote}")
        if note:
            parts.append(f"note: {note}")
        
        return " | ".join(parts)
    
    return str(item).strip()


def list_field_to_text(items):
    if not isinstance(items, list):
        return ""
    
    texts = [item_to_text(item) for item in items]
    texts = [text for text in texts if text]
    
    return "\n".join(texts)


def build_day_embedding_text(day):
    parts = []
    
    date = day.get("date")
    week_id = day.get("week_id")
    short_summary = str(day.get("short_summary", "")).strip()
    
    if date:
        parts.append(f"date: {date}")
    
    if week_id:
        parts.append(f"week_id: {week_id}")
    
    if short_summary:
        parts.append(f"short_summary:\n{short_summary}")
    
    for field in TEXT_FIELDS_FOR_DAY_INDEX:
        field_text = list_field_to_text(day.get(field, []))
        
        if field_text:
            parts.append(f"{field}:\n{field_text}")
    
    return "\n\n".join(parts)

In [11]:
day_index_rows = []

for day in daily_records:
    day_index_rows.append({
        "doc_id": f"day:{day.get('date')}",
        "date": day.get("date"),
        "week_id": day.get("week_id"),
        "source_day_md": day.get("source_day_md"),
        "source_daily_json": day.get("_json_path"),
        "short_summary": day.get("short_summary", ""),
        "embedding_text": build_day_embedding_text(day),
    })

day_index_df = pd.DataFrame(day_index_rows)

day_index_df[["doc_id", "date", "week_id", "short_summary"]].head()

,doc_id,date,week_id,short_summary
0,day:2026-03-29,2026-03-29,2026-W14,"Автор провел день, занимаясь чтением манги, об..."
1,day:2026-03-30,2026-03-30,2026-W14,"Автор активно провёл день, участвовал в хакато..."
2,day:2026-03-31,2026-03-31,2026-W14,"Автор провёл насыщенный день, включая учёбу, ф..."
3,day:2026-04-01,2026-04-01,2026-W14,"Автор провёл активный учебный день, начав с по..."
4,day:2026-04-02,2026-04-02,2026-W14,День начался с дискомфорта из-за боли в животе...


In [12]:
day_index_df["embedding_text_len"] = day_index_df["embedding_text"].str.len()

day_index_df.head()

,doc_id,date,week_id,source_day_md,source_daily_json,short_summary,embedding_text,embedding_text_len
0,day:2026-03-29,2026-03-29,2026-W14,data/processed/days_md/2026-03-29.md,C:\Users\79104\Data Science\Data-Science-\Solo...,"Автор провел день, занимаясь чтением манги, об...",date: 2026-03-29\n\nweek_id: 2026-W14\n\nshort...,1787
1,day:2026-03-30,2026-03-30,2026-W14,data/processed/days_md/2026-03-30.md,C:\Users\79104\Data Science\Data-Science-\Solo...,"Автор активно провёл день, участвовал в хакато...",date: 2026-03-30\n\nweek_id: 2026-W14\n\nshort...,2194
2,day:2026-03-31,2026-03-31,2026-W14,data/processed/days_md/2026-03-31.md,C:\Users\79104\Data Science\Data-Science-\Solo...,"Автор провёл насыщенный день, включая учёбу, ф...",date: 2026-03-31\n\nweek_id: 2026-W14\n\nshort...,2402
3,day:2026-04-01,2026-04-01,2026-W14,data/processed/days_md/2026-04-01.md,C:\Users\79104\Data Science\Data-Science-\Solo...,"Автор провёл активный учебный день, начав с по...",date: 2026-04-01\n\nweek_id: 2026-W14\n\nshort...,3662
4,day:2026-04-02,2026-04-02,2026-W14,data/processed/days_md/2026-04-02.md,C:\Users\79104\Data Science\Data-Science-\Solo...,День начался с дискомфорта из-за боли в животе...,date: 2026-04-02\n\nweek_id: 2026-W14\n\nshort...,1998


In [13]:
sample = day_index_df.sample(1, random_state=42).iloc[0]

print("DATE:", sample["date"])
print("WEEK:", sample["week_id"])
print("LEN:", sample["embedding_text_len"])
print("-" * 80)
print(sample["embedding_text"][:3000])

DATE: 2026-04-02
WEEK: 2026-W14
LEN: 1998
--------------------------------------------------------------------------------
date: 2026-04-02

week_id: 2026-W14

short_summary:
День начался с дискомфорта из-за боли в животе, затем прошел активно благодаря учебной деятельности и общению с мамой. Автор боролся с желанием бросить начатое и переживал последствия своего общения с близкими.

important_moments:
quote: Встал сначала в 5, потом в 7 утра. | note: Раннее пробуждение из-за боли в животе.
quote: Сел за алгосы, потом посидел на хх, поделал немного хакатон и сел за первую лабу Царьковой. | note: Утренняя учебная активность.
quote: Посмотрел четвёртый сезон класса превосходства. | note: Просмотр аниме.
quote: Залипал в Ютубе, а потом пошёл прогуляться. | note: Отдых во второй половине дня.
quote: Сходил на диспансеризацию. | note: Забота о здоровье.
quote: Вечером зачем-то сказал маме о своих планах по учебе. | note: Общение с мамой, попытка поделиться планами.

wins:
quote: Сел за перв

In [14]:
shortest = day_index_df.sort_values("embedding_text_len").head(5)

shortest[["date", "week_id", "embedding_text_len", "short_summary"]]

,date,week_id,embedding_text_len,short_summary
25,2026-04-23,2026-W17,35,
32,2026-04-30,2026-W18,35,
23,2026-04-21,2026-W17,35,
24,2026-04-22,2026-W17,35,
51,2026-05-19,2026-W21,35,


### 5.1. Fallback for failed daily JSON extraction

Некоторые daily JSON почти пустые, потому что LLM отказалась обрабатывать часть дней.

Для raw-first retrieval это не должно полностью удалять день из поиска.

Если `embedding_text` слишком короткий, используем fallback:

```text
date + week_id + first N chars from raw day markdown

In [15]:
MIN_DAY_INDEX_TEXT_LEN = 200
RAW_FALLBACK_CHARS = 2500


def read_day_raw_text(source_day_md: str) -> str:
    path = resolve_source_day_md(source_day_md)
    return path.read_text(encoding="utf-8")


def build_fallback_embedding_text(row) -> str:
    raw_text = read_day_raw_text(row["source_day_md"])
    raw_text = raw_text.strip()

    return "\n\n".join([
        f"date: {row['date']}",
        f"week_id: {row['week_id']}",
        "raw_day_markdown_fallback:",
        raw_text[:RAW_FALLBACK_CHARS],
    ])


day_index_df["used_raw_fallback"] = day_index_df["embedding_text_len"] < MIN_DAY_INDEX_TEXT_LEN

day_index_df.loc[
    day_index_df["used_raw_fallback"],
    "embedding_text"
] = day_index_df[day_index_df["used_raw_fallback"]].apply(
    build_fallback_embedding_text,
    axis=1
)

day_index_df["embedding_text_len"] = day_index_df["embedding_text"].str.len()

day_index_df["used_raw_fallback"].value_counts()

used_raw_fallback
False    61
True     14
Name: count, dtype: int64

In [16]:
row = day_index_df[day_index_df["used_raw_fallback"]].iloc[0]

print("DATE:", row["date"])
print("WEEK:", row["week_id"])
print("LEN:", row["embedding_text_len"])
print("-" * 80)
print(row["embedding_text"][:3000])

DATE: 2026-04-09
WEEK: 2026-W15
LEN: 1914
--------------------------------------------------------------------------------
date: 2026-04-09

week_id: 2026-W15

raw_day_markdown_fallback:

# Diary day: 2026-04-09

Original header: 9 апр. 2026 г. Чт
Week ID: 2026-W15

Снова универ
Устал
Как книга? Всё так же интересно. Возможно, сейчас была
самая сочная сцена, которая отлично раскрывает гг. Ещё меня
бесит Хорикита, потому что она слишком дохрена ожидает от
других, при этом сама идя наперекор логике.
Утром встал, поел и поехал в уник. Я снова стоял, только уже
в экспрессе.
На паре ничего особого не было: поговорили про то, как
закрывается предмет, я себе взял реферат и думаю на некст
пару прийти, рассказать реферат и больше не приходить.
Мы с Андреем решили скипнуть вторую пару и поехать на
Электрозаводской на физру. Поговорили насчёт моего data
science пути. Может быть, я действительно слишком ужко
смотрю. Итогом наших дискуссий стала мысль, что нужно
заниматься нетворкингом и по возможн

## 6. Retrieval

В этом notebook используем day-first retrieval.

Основная схема:

1. Сначала ищем релевантные дни через `day_index_df["embedding_text"]`.
2. Получаем top-k дней.
3. Открываем raw `days_md` только для найденных дней.
4. Уже внутри этих дней ищем evidence-фрагменты.

### 6.1. Retrieval evaluation setup

Фиксируем маленький ручной eval-set.

Оцениваем качество top-k выдачи.

Каждый найденный день размечается вручную:

- 2 = явно релевантно
- 1 = частично релевантно
- 0 = нерелевантно

Метрики:

- precision@k: доля результатов с label >= 1
- strong_precision@k: доля результатов с label == 2
- avg_relevance@k: средняя оценка релевантности

In [17]:
def get_search_results(search_fn, query: str, top_k: int = 5) -> pd.DataFrame:
    results = search_fn(query, top_k=top_k).copy()
    results = results.reset_index(drop=True)
    results["rank"] = results.index + 1

    return results[
        [
            "rank",
            "score",
            "date",
            "week_id",
            "short_summary",
            "used_raw_fallback",
            "source_day_md",
        ]
    ]

In [18]:
def evaluate_manual_labels(manual_labels: list[dict]) -> pd.DataFrame:
    rows = []

    for item in manual_labels:
        labels = item["labels"]
        top_k = item["top_k"]

        labels_array = np.array(labels)

        precision_at_k = (labels_array >= 1).sum() / top_k
        strong_precision_at_k = (labels_array == 2).sum() / top_k
        avg_relevance_at_k = labels_array.mean()

        rows.append({
            "query": item["query"],
            "top_k": top_k,
            "labels": labels,
            "precision_at_k": precision_at_k,
            "strong_precision_at_k": strong_precision_at_k,
            "avg_relevance_at_k": avg_relevance_at_k
        })

    return pd.DataFrame(rows)

### 6.2. TF-IDF baseline for day search

In [19]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=20_000,
    ngram_range=(1, 2)
)

day_texts = day_index_df['embedding_text'].fillna('').tolist()
tfidf_matrix = tfidf_vectorizer.fit_transform(day_texts)
tfidf_matrix.shape

(75, 18596)

Делаем функцию поиска:

query → vector → cosine similarity → top-k days

In [20]:
def search_days_tfidf(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vector = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = day_index_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    
    columns = [
        "score",
        "date",
        "week_id",
        "short_summary",
        "source_day_md",
        "used_raw_fallback",
    ]
    
    return results[columns]

In [21]:
get_search_results(search_days_tfidf, "скука", top_k=5)

,rank,score,date,week_id,short_summary,used_raw_fallback,source_day_md
0,1,0.103215,2026-06-10,2026-W24,Автор провёл день за работой над проектом по M...,False,data/processed/days_md/2026-06-10.md
1,2,0.045505,2026-04-26,2026-W18,"Автор провёл день за игрой в Ведьмака, отмечая...",False,data/processed/days_md/2026-04-26.md
2,3,0.039667,2026-06-13,2026-W24,"Автор провёл день в скуке и усталости, читая и...",False,data/processed/days_md/2026-06-13.md
3,4,0.030772,2026-04-20,2026-W17,"Автор провел занятый и суетливый день, выполня...",False,data/processed/days_md/2026-04-20.md
4,5,0.025401,2026-04-10,2026-W15,Автор переживает сложный период высокой загруж...,False,data/processed/days_md/2026-04-10.md


In [22]:
get_search_results(search_days_tfidf, "ds, машинное обучение, стажировка", top_k=5)

,rank,score,date,week_id,short_summary,used_raw_fallback,source_day_md
0,1,0.084588,2026-04-30,2026-W18,,True,data/processed/days_md/2026-04-30.md
1,2,0.057038,2026-04-27,2026-W18,"День начинается с занятий по DS и лаборатории,...",False,data/processed/days_md/2026-04-27.md
2,3,0.047455,2026-04-11,2026-W15,Автор провёл день за чтением увлекательной кни...,False,data/processed/days_md/2026-04-11.md
3,4,0.039899,2026-04-01,2026-W14,"Автор провёл активный учебный день, начав с по...",False,data/processed/days_md/2026-04-01.md
4,5,0.038021,2026-04-12,2026-W16,"Автор провёл активный выходной день, сочетая о...",False,data/processed/days_md/2026-04-12.md


In [23]:
tfidf_manual_labels = [
    {
        "query": "скука",
        "top_k": 5,
        "labels": [2, 0, 2, 0, 0]
    },
    {
        "query": "ds, машинное обучение, стажировка",
        "top_k": 5,
        "labels": [2, 1, 1, 1, 2]
    },
]



In [24]:
evaluate_manual_labels(tfidf_manual_labels)

,query,top_k,labels,precision_at_k,strong_precision_at_k,avg_relevance_at_k
0,скука,5,"[2, 0, 2, 0, 0]",0.4,0.4,0.8
1,"ds, машинное обучение, стажировка",5,"[2, 1, 1, 1, 2]",1.0,0.4,1.4


TF-IDF нашёл 2 явно релевантных дня из 5, но выдача довольно шумная

### 6.3. Semantic day search

Теперь считаем dense embeddings для `embedding_text`.

В отличие от TF-IDF, semantic embedding должен лучше находить смысловые совпадения.

In [25]:
MODEL_NAME = "intfloat/multilingual-e5-small"

model = SentenceTransformer(MODEL_NAME)
model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [26]:
day_passages = [
    'passage: ' + text 
    for text in day_index_df['embedding_text'].fillna('').tolist()
]

day_embeddings = model.encode(
    day_passages,
    normalize_embeddings=True,
    show_progress_bar=True
)

day_embeddings.shape

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

(75, 384)

In [27]:
def search_days_embeddings(query: str, top_k: int = 5) -> pd.DataFrame:
    query_embedding = model.encode(
        ["query: " + query],
        normalize_embeddings=True,
    )[0]

    scores = day_embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = day_index_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    results = results.reset_index(drop=True)
    results["rank"] = results.index + 1

    return results[
        [
            "rank",
            "score",
            "date",
            "week_id",
            "short_summary",
            "used_raw_fallback",
            "source_day_md",
        ]
    ]

In [28]:
search_days_embeddings("скука", top_k=5)

,rank,score,date,week_id,short_summary,used_raw_fallback,source_day_md
0,1,0.812169,2026-05-28,2026-W22,"День прошел скучно и однообразно на работе, ве...",False,data/processed/days_md/2026-05-28.md
1,2,0.810467,2026-05-26,2026-W22,День начался с поиска интереса в скуке и улучш...,False,data/processed/days_md/2026-05-26.md
2,3,0.808702,2026-06-13,2026-W24,"Автор провёл день в скуке и усталости, читая и...",False,data/processed/days_md/2026-06-13.md
3,4,0.799535,2026-03-31,2026-W14,"Автор провёл насыщенный день, включая учёбу, ф...",False,data/processed/days_md/2026-03-31.md
4,5,0.799104,2026-06-10,2026-W24,Автор провёл день за работой над проектом по M...,False,data/processed/days_md/2026-06-10.md


In [29]:
search_days_embeddings("ds, машинное обучение, стажировка", top_k=5)

,rank,score,date,week_id,short_summary,used_raw_fallback,source_day_md
0,1,0.870698,2026-04-01,2026-W14,"Автор провёл активный учебный день, начав с по...",False,data/processed/days_md/2026-04-01.md
1,2,0.864158,2026-05-13,2026-W20,"Автор проснулся поздно, проспав 10 часов, и пр...",False,data/processed/days_md/2026-05-13.md
2,3,0.858646,2026-04-27,2026-W18,"День начинается с занятий по DS и лаборатории,...",False,data/processed/days_md/2026-04-27.md
3,4,0.854165,2026-05-14,2026-W20,"Автор провел день, занимаясь выполнением задан...",False,data/processed/days_md/2026-05-14.md
4,5,0.851404,2026-04-03,2026-W14,"Автор провёл насыщенный день, совмещая учёбу, ...",False,data/processed/days_md/2026-04-03.md


In [30]:
embeddings_manual_labels = [
    {
        "query": "скука",
        "top_k": 5,
        "labels": [2, 2, 2, 0, 2]
    },
    {
        "query": "ds, машинное обучение, стажировка",
        "top_k": 5,
        "labels": [2, 2, 2, 1, 2]
    },
]

In [31]:
evaluate_manual_labels(embeddings_manual_labels)

,query,top_k,labels,precision_at_k,strong_precision_at_k,avg_relevance_at_k
0,скука,5,"[2, 2, 2, 0, 2]",0.8,0.8,1.6
1,"ds, машинное обучение, стажировка",5,"[2, 2, 2, 1, 2]",1.0,0.8,1.8


На мини-eval из двух запросов semantic embeddings показали более качественную top-k выдачу, чем TF-IDF.
embeddings лучше поднимают наверх сильные совпадения, а не просто частично связанные дни.

### 6.35 Retrieval parameter experiments


Сравниваем:

- `top_k_days`
- `top_k_chunks`
- `chunk_size`
- `overlap`

Каждый найденный chunk размечается вручную:

- 2 = явно релевантный evidence
- 1 = частично релевантный / полезный
- 0 = шум

### 6.4. Local evidence search inside top-k days

Теперь переходим от поиска дней к поиску evidence.

Stage 1 нашёл top-k релевантных дней через `day_index`.

Stage 2 открывает raw `days_md` только для этих дней и ищет внутри них конкретные фрагменты.

In [32]:
def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # remove markdown/html page markers from PDF extraction
    text = re.sub(r"<!--\s*page\s+\d+\s*-->", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def chunk_text_by_chars(
    text: str,
    chunk_size: int = 600,
    overlap: int = 100,
    min_chunk_len: int = 350,
) -> list[str]:
    text = normalize_text(text)

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    # If the last chunk is too short, merge it into the previous one.
    # This avoids weak 100–300 char chunks with unstable semantic scores.
    if len(chunks) >= 2 and len(chunks[-1]) < min_chunk_len:
        chunks[-2] = (chunks[-2] + "\n\n" + chunks[-1]).strip()
        chunks = chunks[:-1]

    # If there is only one chunk and it is short, keep it.
    # A short day is still better than losing the whole day.
    return chunks

In [33]:
DAY_HEADER_SKIP_CHARS = 130

def build_local_chunks_from_days(
    days_df: pd.DataFrame,
    chunk_size: int = 600,
    overlap: int = 100,
    skip_start_chars: int = DAY_HEADER_SKIP_CHARS,
) -> pd.DataFrame:
    chunk_rows = []

    for _, row in days_df.iterrows():
        raw_text = read_day_raw_text(row["source_day_md"])
        raw_text = raw_text[skip_start_chars:].strip()

        chunks = chunk_text_by_chars(
            raw_text,
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk_index, chunk in enumerate(chunks):
            chunk_rows.append({
                "doc_id": f"local_chunk:{row['date']}:{chunk_index:03d}",
                "date": row["date"],
                "week_id": row["week_id"],
                "source_day_md": row["source_day_md"],
                "chunk_index": chunk_index,
                "text": chunk,
                "text_len": len(chunk),
            })

    return pd.DataFrame(chunk_rows)

In [34]:
def search_evidence_chunks(query: str, top_k_days: int = 5, top_k_chunks: int = 5) -> pd.DataFrame:
    top_days = search_days_embeddings(query, top_k=top_k_days)

    local_chunks_df = build_local_chunks_from_days(
        top_days,
        chunk_size=600,
        overlap=100,
    )

    if local_chunks_df.empty:
        return local_chunks_df

    chunk_passages = [
        "passage: " + text
        for text in local_chunks_df["text"].fillna("").tolist()
    ]

    chunk_embeddings = model.encode(
        chunk_passages,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    query_embedding = model.encode(
        ["query: " + query],
        normalize_embeddings=True,
    )[0]

    scores = chunk_embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k_chunks]

    results = local_chunks_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    results = results.reset_index(drop=True)
    results["rank"] = results.index + 1

    return results[
        [
            "rank",
            "score",
            "date",
            "week_id",
            "chunk_index",
            "text_len",
            "source_day_md",
            "text",
        ]
    ]

Первый тест

In [35]:
evidence_results = search_evidence_chunks(
    query="скука",
    top_k_days=5,
    top_k_chunks=5,
)

evidence_results[["rank", "score", "date", "week_id", "chunk_index", "text_len", "source_day_md"]]

,rank,score,date,week_id,chunk_index,text_len,source_day_md
0,1,0.789228,2026-06-10,2026-W24,1,845,data/processed/days_md/2026-06-10.md
1,2,0.789134,2026-05-26,2026-W22,1,600,data/processed/days_md/2026-05-26.md
2,3,0.777523,2026-03-31,2026-W14,1,776,data/processed/days_md/2026-03-31.md
3,4,0.772953,2026-06-13,2026-W24,2,456,data/processed/days_md/2026-06-13.md
4,5,0.772420,2026-05-28,2026-W22,0,475,data/processed/days_md/2026-05-28.md


Посмотрим найденные фрагменты

In [36]:
for _, row in evidence_results.iterrows():
    print("=" * 100)
    print("RANK:", row["rank"])
    print("SCORE:", round(row["score"], 4))
    print("DATE:", row["date"])
    print("WEEK:", row["week_id"])
    print("CHUNK:", row["chunk_index"])
    print("SOURCE:", row["source_day_md"])
    print("-" * 100)
    print(row["text"])
    print()

RANK: 1
SCORE: 0.7892
DATE: 2026-06-10
WEEK: 2026-W24
CHUNK: 1
SOURCE: data/processed/days_md/2026-06-10.md
----------------------------------------------------------------------------------------------------
пошел на турники. Сегодня хорошо подзабил
плечи. Параллельно занимался инвестициями.
Уже дома я сел играть в новеллу. Ещё раз напомню: мне
очень понравилось. Не то, что бы там сильный

интеллектуальный жет, нет, он очень трогательный и
эмоциональный, цепляющий и приятный.
Потом снова посидел над проектом, сходи в магазин, потом
мучал себя скукой - мне действительно было нечего делать: я
не хотел играть, читать новеллу, мангу, заниматься проектом,
так что у меня просто ничего не оставалось. Я просто гулял,
слушал музыку.
Даже вышел на балкон и очень атмосферненько слушал
осты, попивая чай. Може

сто гулял,
слушал музыку.
Даже вышел на балкон и очень атмосферненько слушал
осты, попивая чай. Может, так и должно быть?
Нет столько тревоги, есть время, да и настроение хорошее. Я
счастли

### 6.7. Build evidence context

Теперь превращаем найденные chunks в компактный evidence context

In [37]:
def build_evidence_context(
    query: str,
    evidence_df: pd.DataFrame,
    max_chunk_chars: int = 900,
) -> str:
    parts = []

    parts.append(f"Question: {query}")
    parts.append("")
    parts.append("Retrieved evidence chunks:")
    parts.append("")

    for _, row in evidence_df.iterrows():
        text = str(row["text"]).strip()

        if len(text) > max_chunk_chars:
            text = text[:max_chunk_chars].rstrip() + "..."

        parts.append(f"[{row['rank']}] date: {row['date']}")
        parts.append(f"week_id: {row['week_id']}")
        parts.append(f"score: {row['score']:.4f}")
        parts.append(f"source_day_md: {row['source_day_md']}")
        parts.append(f"chunk_index: {row['chunk_index']}")
        parts.append("text:")
        parts.append(text)
        parts.append("")

    return "\n".join(parts)

In [38]:
query = "скука"

evidence_results = search_evidence_chunks(
    query=query,
    top_k_days=5,
    top_k_chunks=5,
)

evidence_context = build_evidence_context(
    query=query,
    evidence_df=evidence_results,
)

print(evidence_context)

Question: скука

Retrieved evidence chunks:

[1] date: 2026-06-10
week_id: 2026-W24
score: 0.7892
source_day_md: data/processed/days_md/2026-06-10.md
chunk_index: 1
text:
пошел на турники. Сегодня хорошо подзабил
плечи. Параллельно занимался инвестициями.
Уже дома я сел играть в новеллу. Ещё раз напомню: мне
очень понравилось. Не то, что бы там сильный

интеллектуальный жет, нет, он очень трогательный и
эмоциональный, цепляющий и приятный.
Потом снова посидел над проектом, сходи в магазин, потом
мучал себя скукой - мне действительно было нечего делать: я
не хотел играть, читать новеллу, мангу, заниматься проектом,
так что у меня просто ничего не оставалось. Я просто гулял,
слушал музыку.
Даже вышел на балкон и очень атмосферненько слушал
осты, попивая чай. Може

сто гулял,
слушал музыку.
Даже вышел на балкон и очень атмосферненько слушал
осты, попивая чай. Может, так и должно быть?
Нет столько тревоги, есть время, да и настроение хорошее. Я
счастлив.
Здоровья здоровья, здоровья и силы 

### 6.8. Build ask_history

Теперь собираем prompt для будущего `ask_history`.

На вход:

- вопрос пользователя;
- retrieved evidence context из raw day markdown chunks.

In [39]:
def build_ask_history_prompt(query: str, evidence_context: str) -> str:
    return f"""
Твоя роль — вдумчивый и честный собеседник, который помогает мне разбирать мои дневниковые записи.

Ты получил не весь дневник, а только найденные фрагменты, которые retrieval посчитал релевантными вопросу пользователя.

Твоя задача — не просто пересказать найденное, а помочь мне лучше понять, что в этих фрагментах происходит.

Что нужно делать:
— кратко выделять суть найденных фрагментов;
— отделять факты от эмоций и интерпретаций;
— замечать повторяющиеся паттерны, страхи, триггеры, желания и внутренние противоречия;
— показывать, что, судя по фрагментам, меня реально задевало и почему;
— помогать понять, где я мог накручивать себя, где избегал чего-то, а где смотрел на ситуацию трезво;
— давать спокойный, честный и глубокий разбор без осуждения;
— в конце делать короткий практический вывод: что стоит понять, отпустить, принять или проверить дальше.

Правила ответа:
— не отвечай шаблонно и поверхностно;
— не обесценивай мои эмоции;
— не поддакивай мне автоматически, если выводы в записи выглядят спорно;
— не давай банальные советы;
— не ставь диагнозы;
— не выдумывай факты, которых нет в найденных фрагментах;
— если данных мало, прямо скажи, что вывод предварительный;
— пиши понятно, структурно и по делу;
— если в найденных фрагментах есть важный повторяющийся паттерн, обязательно укажи на него;
— если я эмоционален, сначала помоги разложить ситуацию по полочкам, а потом уже анализируй;
— основной текст должен быть приятным для чтения, а не похожим на формальный отчёт;


Формат ответа:

# Ответ

Дай связный главный вывод по вопросу. 
Не сухой summary, а нормальный человеческий разбор: что, похоже, происходит, какая общая линия видна, что объединяет найденные дни.

## Суть найденных фрагментов

Кратко объясни, о чём эти фрагменты по сути.

## Что здесь происходит эмоционально и психологически

Разбери, какие эмоции, напряжения, желания или внутренние конфликты видны в найденных фрагментах.

Пиши осторожно: не как диагноз, а как интерпретацию по дневнику.

## На что стоит обратить внимание

Выдели 2–4 важных наблюдения.

## Возможные искажения или самообман

Покажи, где я мог:
— слишком жёстко себя оценивать;
— преувеличивать проблему;
— избегать чего-то;
— путать факт и интерпретацию.

Если по найденным фрагментам этого не видно, так и напиши.

## Практический вывод

Коротко: что из этого стоит понять, отпустить, принять или проверить дальше.

## Один короткий вопрос для самоанализа

Задай один вопрос, который логично вытекает из разбора.

## Цитаты и evidence

Этот блок обязателен. Не пропускай его. Если нет цитат, пиши в этом блоке, что нет evidence.

Дай 3–7 пунктов.

Формат каждого пункта строго такой:

- YYYY-MM-DD — "короткая цитата"

Вопрос пользователя:
{query}

Найденные фрагменты дневника:
{evidence_context}
""".strip()

In [40]:
ask_history_prompt = build_ask_history_prompt(
    query=query,
    evidence_context=evidence_context,
)

### 6.9. Retrieval check with API

In [41]:
load_dotenv()

GIGACHAT_AUTH_KEY = os.getenv("GIGACHAT_AUTH_KEY")
GIGACHAT_MODEL_DAILY = os.getenv("GIGACHAT_MODEL_DAILY")

print("GIGACHAT_AUTH_KEY exists:", bool(GIGACHAT_AUTH_KEY))
print("GIGACHAT_MODEL_DAILY:", GIGACHAT_MODEL_DAILY)

GIGACHAT_AUTH_KEY exists: True
GIGACHAT_MODEL_DAILY: GigaChat-2


In [42]:
with GigaChat(
    credentials=GIGACHAT_AUTH_KEY,
    model=GIGACHAT_MODEL_DAILY,
    verify_ssl_certs=False,
) as giga:
    response = giga.chat("Ответь одним коротким предложением: API работает?")

print(response.choices[0].message.content)

API функционирует нормально.


Создаём маленькую функцию для вызова GigaChat

In [43]:
def call_gigachat(prompt: str) -> str:
    with GigaChat(
        credentials=GIGACHAT_AUTH_KEY,
        model=GIGACHAT_MODEL_DAILY,
        verify_ssl_certs=False,
    ) as giga:
        response = giga.chat(prompt)

    return response.choices[0].message.content

На мини-eval из двух запросов semantic embeddings показали более качественную top-k выдачу, чем TF-IDF.
embeddings лучше поднимают наверх сильные совпадения, а не просто частично связанные дни.

In [44]:
ask_history_answer = call_gigachat(ask_history_prompt)

print(ask_history_answer)#скука

# Ответ

Судя по дневниковым записям, скука становится частым спутником автора, особенно когда нет чётких планов или занятий. Автор описывает своё состояние скуки нейтрально, без явной негативной окраски, скорее наблюдая за самим процессом ожидания момента занятия чем-либо интересным. Однако именно эта пустота заполняется меланхоличными прогулками, прослушиванием музыки и чаепитиями на балконе, создающими ощущение атмосферы, которую автор воспринимает спокойно и даже радостно.

## Суть найденных фрагментов

Автор чаще всего сталкивается с ощущением скуки после завершения активного дня или выполнения необходимых дел. Обычно скука проявляется тогда, когда отсутствуют привычные источники развлечений (чтение, игра, проекты), однако она редко вызывает тревогу или негативные переживания. Напротив, иногда скука воспринимается положительно, поскольку позволяет отдохнуть от напряжённого ритма жизни.

## Что здесь происходит эмоционально и психологически

Скука часто сопровождается расслабленным

Ручная оценка:

- source_fidelity: 1.5/2
- evidence_quality: 2/2
- scope_control: 1/2
- usefulness: 2/2
- limitations: 1.5/2

Total: 8/10

Notes:
- Ответ в целом полезный и evidence-based.
- Есть противоречие: в интерпретации модель пишет "часто", но в limitations признаёт, что частота неясна.

### 6.10. Ask history mini eval

Проверяем текущий ask_history pipeline на нескольких типах вопросов.

Цель — понять не среднее качество модели, а слабые места retrieval:

- какие темы находятся хорошо;
- где day_index не вытаскивает нужные дни;
- где chunks дают шум;
- где LLM делает вывод шире evidence.

In [45]:
test_queries = [
    "Когда я чувствую скуку?",
    "Как я чувствую себя при работе в ML?",
    "Когда я обычно переживаю?",
    "Есть ли у меня прогресс в спорте?",
    "Как я разбираюсь с усталостью?",
    "Как я обычно веду себя с друзьями?",
    "Какое моё самое нелюбимое чувство?",
]

In [46]:
def run_ask_history_query(
    question: str,
    top_k_days: int = 5,
    top_k_chunks: int = 5,
    max_chunk_chars: int = 900,
) -> dict:
    evidence_results = search_evidence_chunks(
        query=question,
        top_k_days=top_k_days,
        top_k_chunks=top_k_chunks,
    )

    evidence_context = build_evidence_context(
        query=question,
        evidence_df=evidence_results,
        max_chunk_chars=max_chunk_chars,
    )

    prompt = build_ask_history_prompt(
        query=question,
        evidence_context=evidence_context,
    )

    answer = call_gigachat(prompt)

    return {
        "question": question,
        "evidence_results": evidence_results,
        "evidence_context": evidence_context,
        "prompt": prompt,
        "answer": answer,
    }

In [47]:
result = run_ask_history_query(
    question="Когда я чувствую скуку?",
    top_k_days=7,
    top_k_chunks=10,
)

print(result["answer"])

# Ответ

Судя по найденным записям, скука становится частым спутником автора, особенно тогда, когда жизнь теряет привычный ритм активности или новые впечатления становятся редкостью. Фрагменты показывают, что скука сопровождается чувством внутреннего беспокойства, усталости и недостатком мотивации, хотя внешне автор пытается скрывать свое состояние чрезмерной занятостью и увлеченностью повседневными обязанностями.

## Суть найденных фрагментов

Автор описывает скуку через образы бега белки в колесе, ощущения пустоты и отсутствия направления в жизни. Скука возникает не только из-за недостатка внешних стимулов, но и вследствие внутренней неудовлетворенности собственным состоянием и образом жизни. Несмотря на желание развиваться и заниматься продуктивными занятиями (например, решать алгоритмы, тренироваться), часто проявляется склонность искать утешение в простых удовольствиях, таких как потребление контента или игры.

## Что здесь происходит эмоционально и психологически

Фрагменты отраж

Вывод приятно читать + есть evidence и встроенные цитаты

## 7. Retrieval experiments

Цель: подобрать параметры, при которых `ask_history` получает хорошие evidence chunks.

Сравниваем:

- `top_k_days` — сколько дней достаём на первом этапе;
- `top_k_chunks` — сколько raw chunks отдаём в prompt;
- `chunk_size` — размер локального chunk внутри найденных дней;
- `overlap` — пересечение между chunks.

заменим текущую функцию search_evidence_chunks на версию, где можно менять параметры

In [49]:
def search_evidence_chunks(
    query: str,
    top_k_days: int = 5,
    top_k_chunks: int = 5,
    chunk_size: int = 600,
    overlap: int = 100,
) -> pd.DataFrame:
    top_days = search_days_embeddings(
        query=query,
        top_k=top_k_days,
    )

    local_chunks_df = build_local_chunks_from_days(
        days_df=top_days,
        chunk_size=chunk_size,
        overlap=overlap,
    )

    if local_chunks_df.empty:
        return local_chunks_df

    chunk_passages = [
        "passage: " + text
        for text in local_chunks_df["text"].fillna("").tolist()
    ]

    chunk_embeddings = model.encode(
        chunk_passages,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    query_embedding = model.encode(
        ["query: " + query],
        normalize_embeddings=True,
    )[0]

    scores = chunk_embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k_chunks]

    results = local_chunks_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    results = results.reset_index(drop=True)
    results["rank"] = results.index + 1

    results["query"] = query
    results["top_k_days"] = top_k_days
    results["top_k_chunks"] = top_k_chunks
    results["chunk_size"] = chunk_size
    results["overlap"] = overlap

    return results[
        [
            "query",
            "rank",
            "score",
            "date",
            "week_id",
            "chunk_index",
            "text_len",
            "top_k_days",
            "top_k_chunks",
            "chunk_size",
            "overlap",
            "source_day_md",
            "text",
        ]
    ]

In [52]:
retrieval_test_queries = [
    "Когда я чувствую скуку?",
    "Какие у меня есть интересы?",
    "Когда я обычно болею?",
    "Когда я чувствую радость?",
]

In [68]:
retrieval_configs = [
    {
        "config_id": "k5_c600",
        "top_k_days": 5,
        "top_k_chunks": 5,
        "chunk_size": 600,
        "overlap": 100,
    },
    {
        "config_id": "k7_c800",
        "top_k_days": 7,
        "top_k_chunks": 7,
        "chunk_size": 800,
        "overlap": 150,
    },
    {
        "config_id": "k7_c500",
        "top_k_days": 7,
        "top_k_chunks": 7,
        "chunk_size": 500,
        "overlap": 100,
    },
    {
        "config_id": "k12_c600",
        "top_k_days": 12,
        "top_k_chunks": 12,
        "chunk_size": 600,
        "overlap": 100,
    },
]

In [69]:
experiment_rows = []

for query in retrieval_test_queries:
    for config in retrieval_configs:
        results = search_evidence_chunks(
            query=query,
            top_k_days=config["top_k_days"],
            top_k_chunks=config["top_k_chunks"],
            chunk_size=config["chunk_size"],
            overlap=config["overlap"],
        ).copy()

        results["config_id"] = config["config_id"]

        experiment_rows.append(results)

retrieval_experiments_df = pd.concat(
    experiment_rows,
    ignore_index=True,
)

retrieval_experiments_df.shape

(124, 14)

In [70]:
def make_text_preview(text: str, max_chars: int = 350) -> str:
    text = str(text).replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    if len(text) > max_chars:
        return text[:max_chars].rstrip() + "..."

    return text

In [71]:
retrieval_labeling_df = retrieval_experiments_df.copy()

retrieval_labeling_df["text_preview"] = retrieval_labeling_df["text"].apply(
    make_text_preview
)

retrieval_labeling_df["label"] = None
retrieval_labeling_df["comment"] = ""

retrieval_labeling_df[
    [
        "query",
        "config_id",
        "rank",
        "score",
        "date",
        "chunk_index",
        "text_len",
        "label",
        "comment",
        "text_preview",
    ]
]

,query,config_id,rank,score,date,chunk_index,text_len,label,comment,text_preview
0,Когда я чувствую скуку?,k5_c600,1,0.825175,2026-06-12,4,467,None,,"вда меня бесит, что они уж слишком часто упоми..."
1,Когда я чувствую скуку?,k5_c600,2,0.820760,2026-05-26,1,600,None,,"потребляю контент к которому привык, но уже в ..."
2,Когда я чувствую скуку?,k5_c600,3,0.819570,2026-06-12,3,600,None,,воим принципам и не меняй их на одобрение друг...
3,Когда я чувствую скуку?,k5_c600,4,0.807805,2026-06-12,1,599,None,,"письмо автора и великолепную постановку. Да, и..."
4,Когда я чувствую скуку?,k5_c600,5,0.806840,2026-03-31,1,776,None,,тоговый тест. Так что не зря готовился. Расска...
...,...,...,...,...,...,...,...,...,...,...
119,Когда я чувствую радость?,k12_c600,8,0.820660,2026-06-04,2,933,None,,ома читал новеллу и обсудил с мамой свою жизнь...
120,Когда я чувствую радость?,k12_c600,9,0.820530,2026-04-25,2,374,None,,После собеса буду отдыхать. Хотя кого я обманы...
121,Когда я чувствую радость?,k12_c600,10,0.818309,2026-06-04,1,599,None,,"м вижу, почему у меня нет мотивации: весь дофа..."
122,Когда я чувствую радость?,k12_c600,11,0.812177,2026-05-31,1,585,None,,"едь действительно, всё же просто: отдыхай, дел..."


In [73]:
retrieval_labeling_df.to_csv('retrieval_labeling_df.csv', index=False)

Разметку на 132 строки сделаю через ИИ

In [74]:
retrieval_labeling_df_labeled = pd.read_csv('retrieval_labeling_df_labeled.csv')

In [75]:
def dcg_at_k(labels: list[int]) -> float:
    labels_array = np.array(labels)

    gains = (2 ** labels_array) - 1
    discounts = np.log2(np.arange(len(labels_array)) + 2)

    return float(np.sum(gains / discounts))


def ndcg_at_k(labels: list[int]) -> float:
    actual_dcg = dcg_at_k(labels)

    ideal_labels = sorted(labels, reverse=True)
    ideal_dcg = dcg_at_k(ideal_labels)

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg


def evaluate_labeled_retrieval(df: pd.DataFrame) -> pd.DataFrame:
    labeled_df = df.dropna(subset=["label"]).copy()
    labeled_df["label"] = labeled_df["label"].astype(int)

    group_cols = [
        "query",
        "config_id",
        "top_k_days",
        "top_k_chunks",
        "chunk_size",
        "overlap",
    ]

    rows = []

    for group_values, group in labeled_df.groupby(group_cols):
        group = group.sort_values("rank")

        labels = group["label"].tolist()
        labels_array = np.array(labels)

        weak_precision_at_k = (labels_array >= 1).sum() / len(labels_array)
        strong_precision_at_k = (labels_array == 2).sum() / len(labels_array)
        avg_relevance_at_k = labels_array.mean()
        ndcg = ndcg_at_k(labels)

        rows.append({
            "query": group_values[0],
            "config_id": group_values[1],
            "top_k_days": group_values[2],
            "top_k_chunks": group_values[3],
            "chunk_size": group_values[4],
            "overlap": group_values[5],
            "labeled_k": len(labels),
            "labels": labels,
            "weak_precision_at_k": weak_precision_at_k,
            "strong_precision_at_k": strong_precision_at_k,
            "avg_relevance_at_k": avg_relevance_at_k,
            "ndcg_at_k": ndcg,
            "unique_dates_at_k": group["date"].nunique(),
            "avg_text_len": group["text_len"].mean(),
        })

    return pd.DataFrame(rows)

In [76]:
retrieval_scores_df = evaluate_labeled_retrieval(retrieval_labeling_df_labeled)
config_summary_df = (
    retrieval_scores_df
    .groupby(["config_id", "top_k_days", "top_k_chunks", "chunk_size", "overlap"])
    .agg(
        mean_ndcg_at_k=("ndcg_at_k", "mean"),
        mean_strong_precision_at_k=("strong_precision_at_k", "mean"),
        mean_weak_precision_at_k=("weak_precision_at_k", "mean"),
        mean_avg_relevance_at_k=("avg_relevance_at_k", "mean"),
        mean_unique_dates_at_k=("unique_dates_at_k", "mean"),
        mean_text_len=("avg_text_len", "mean"),
        queries_count=("query", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["mean_ndcg_at_k", "mean_strong_precision_at_k", "mean_avg_relevance_at_k"],
        ascending=False,
    )
)

config_summary_df

,config_id,top_k_days,top_k_chunks,chunk_size,overlap,mean_ndcg_at_k,mean_strong_precision_at_k,mean_weak_precision_at_k,mean_avg_relevance_at_k,mean_unique_dates_at_k,mean_text_len,queries_count
1,k5_c600,5,5,600,100,0.963109,0.450000,0.800000,1.250000,3.50,598.650000,4
0,k12_c600,12,12,600,100,0.909461,0.458333,0.875000,1.333333,8.00,627.937500,4
2,k7_c500,7,7,500,100,0.853490,0.500000,0.857143,1.357143,4.75,564.500000,4
3,k7_c800,7,7,800,150,0.822783,0.464286,0.821429,1.285714,5.25,756.678571,4


Разные конфиги хороши для разных режимов:
- k5_c600 - лучший по качеству ранжирования
- k12_c600 - лучший для полноценного ask_history, он чуть хуже ранжирует, чем k5_c600, но зато даёт больше полезного материала
- k7_c500 — самый плотный по сильным evidence. Похоже, chunks по 500 символов дают более точные фрагменты: меньше лишнего контекста, больше конкретики.

В DiaryLens вопросы зачастую подразумевают глубокий анализ на большом числе дней, поэтому k12_c600 - лучший баланс для текущего формата ответа. В дальнейшем можно будет сделать отдельные модели под анализ более большого или маленького числа дней. Но пока у модели будут такие параметры:

DEFAULT_TOP_K_DAYS = 12

DEFAULT_TOP_K_CHUNKS = 12

DEFAULT_CHUNK_SIZE = 600

DEFAULT_OVERLAP = 100

DEFAULT_MAX_CHUNK_CHARS = 1200

## 8. Retrieval experiment conclusion

По результатам ручной разметки `k12_c600` выбран как основной режим для `ask_history`.

Причина:

- даёт достаточно evidence для развёрнутого ответа;
- имеет хороший `mean_ndcg_at_k`;
- показывает лучший `mean_weak_precision_at_k`;
- лучше подходит для ответа с цитатами и анализом по нескольким дням.

Дополнительные режимы:

- `k5_c600` — compact/debug mode;
- `k7_c500` — precision mode для более коротких chunks